In [2]:
import os
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve

# ============================================================
# CONFIGURATION
# ============================================================
DATA_PATH = "../datasets/schaefcomb_Wang2023Simple_dfschizo.tsv"
CONF_METHOD = "regression" 
INCLUDE_CONFOUNDS_IN_MODEL = False 
# PAS DE PCA ICI

# ============================================================
# 1) CHARGEMENT ET PRÉPARATION
# ============================================================
print(f"--- Chargement : {DATA_PATH} ---")
df_all = pd.read_csv(DATA_PATH, sep="\t")

mask_loso = (
    df_all["site_id"].isin(["ds000030", "ds_cobre", "ds004302"])
    & df_all["diagnosis"].isin(["CONTROL", "SCHZ"])
)
df_loso = df_all.loc[mask_loso].copy()

# Variables cliniques
df_loso["age"] = df_loso["age"].astype(float)
df_loso["gender_num"] = (df_loso["gender"] == "M").astype(float)

if "mean_fd" not in df_loso.columns:
    df_loso["mean_fd"] = 0.0
else:
    df_loso["mean_fd"] = df_loso["mean_fd"].astype(float)
    if df_loso["mean_fd"].isna().any():
        df_loso["mean_fd"] = df_loso["mean_fd"].fillna(df_loso["mean_fd"].median())

y_loso_all = (df_loso["diagnosis"] == "SCHZ").astype(int)
groups_loso = df_loso["site_id"].values 

# ============================================================
# 2) CONNECTOMES
# ============================================================
corr_cols_all = [c for c in df_loso.columns if c.startswith("corr_")]
X_temp = df_loso[corr_cols_all]
corr_cols = X_temp.columns[X_temp.notna().any()].tolist()

imputer = SimpleImputer(strategy="mean")
X_loso_imp = pd.DataFrame(
    imputer.fit_transform(df_loso[corr_cols]), 
    index=df_loso.index, columns=corr_cols
)

# ============================================================
# 3) FONCTIONS (CORRECTION + MLP RAW)
# ============================================================

def residualize_confounds(X_train, X_test, conf_train, conf_test):
    C_train = np.column_stack([np.ones(len(conf_train)), conf_train.values])
    C_test  = np.column_stack([np.ones(len(conf_test)),  conf_test.values])
    B = np.linalg.pinv(C_train).dot(X_train.values)
    return (pd.DataFrame(X_train.values - C_train.dot(B), index=X_train.index, columns=X_train.columns),
            pd.DataFrame(X_test.values - C_test.dot(B), index=X_test.index, columns=X_test.columns))

def get_model_raw_mlp():
    """Pipeline : Scaler -> MLP (sur 93k features)"""
    
    # MLPClassifier
    # solver='adam' est bien pour les gros volumes de données
    # early_stopping=True est CRITIQUE ici pour éviter l'overfitting immédiat
    mlp = MLPClassifier(
        solver='adam', 
        max_iter=500, # On limite les itérations car c'est lourd
        early_stopping=True, 
        validation_fraction=0.1, # 10% du train utilisé pour vérifier l'overfitting
        n_iter_no_change=5, 
        random_state=42
    )
    
    # Grille adaptée "High Dim Low Sample"
    # On force des architectures très petites (bottleneck) 
    # et une régularisation (alpha) très forte.
    clf = GridSearchCV(
        estimator=mlp,
        param_grid={
            'hidden_layer_sizes': [(10,), (30,)], # Très peu de neurones pour compresser l'info
            'alpha': [1, 10, 50], # Régularisation L2 MASSIVE (par défaut c'est 0.0001)
            'learning_rate_init': [0.001]
        },
        cv=3, # CV réduit à 3 folds pour gagner du temps de calcul
        scoring='roc_auc',
        n_jobs=-1
    )
    
    # Note: Pas de PCA
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])

# ============================================================
# 4) BOUCLE LOSO
# ============================================================
print("\n" + "="*60)
print(f"DÉBUT LOSO (RAW DATA - NO PCA) | MLP Neural Network")
print("ATTENTION : Le temps de calcul sera plus long.")
print("="*60)

logo = LeaveOneGroupOut()
loso_results = []
feature_weights_list = []

for i, (train_idx, test_idx) in enumerate(logo.split(X_loso_imp, y_loso_all, groups=groups_loso)):
    site_test_name = groups_loso[test_idx][0]
    print(f"\n🔹 SITE TEST : {site_test_name}")
    
    X_tr, X_te = X_loso_imp.iloc[train_idx], X_loso_imp.iloc[test_idx]
    y_tr, y_te = y_loso_all.iloc[train_idx], y_loso_all.iloc[test_idx]
    
    conf_vars = ["age", "gender_num", "mean_fd"]
    c_tr, c_te = df_loso.iloc[train_idx][conf_vars], df_loso.iloc[test_idx][conf_vars]

    if CONF_METHOD == "regression":
        X_tr_cl, X_te_cl = residualize_confounds(X_tr, X_te, c_tr, c_te)
    else:
        X_tr_cl, X_te_cl = X_tr, X_te

    # Fit (Directement sur les 93k features)
    pipeline = get_model_raw_mlp()
    pipeline.fit(X_tr_cl, y_tr)
    
    clf_obj = pipeline.named_steps['clf']
    best_hidden = clf_obj.best_params_['hidden_layer_sizes']
    best_alpha = clf_obj.best_params_['alpha']
    auc_int = clf_obj.best_score_
    
    y_prob = pipeline.predict_proba(X_te_cl)[:, 1]
    auc_ext = roc_auc_score(y_te, y_prob)
    brier = brier_score_loss(y_te, y_prob)
    
    print(f"   Hidden: {best_hidden}, Alpha: {best_alpha} | AUC CV: {auc_int:.3f} | AUC Ext: {auc_ext:.3f}")

    loso_results.append({
        "site": site_test_name, "auc_int": auc_int, "auc_ext": auc_ext, 
        "delta": auc_int - auc_ext, "brier": brier
    })

    # --- EXTRACTION IMPORTANCE ---
    # Ici c'est plus simple qu'avec PCA : on prend directement les poids d'entrée
    # Matrice des poids : (n_features, n_hidden_neurons)
    # On fait la somme des valeurs absolues vers tous les neurones cachés
    W_input = clf_obj.best_estimator_.coefs_[0] 
    raw_importance = np.sum(np.abs(W_input), axis=1) # (93000,)
    
    feature_weights_list.append(pd.DataFrame({
        "feature": X_tr_cl.columns, 
        "weight": raw_importance, 
        "test_site": site_test_name
    }))

# Tableau Résumé
df_res = pd.DataFrame(loso_results)
print("\n--- RÉSULTATS RAW MLP ---")
print(df_res[["site", "auc_int", "auc_ext", "delta", "brier"]].round(3))
print(f"Moyenne AUC Externe : {df_res['auc_ext'].mean():.3f}")

# ============================================================
# 5) SANITY CHECK (RANDOM)
# ============================================================
print("\n" + "="*60)
print(f"DÉBUT SANITY CHECK")
print("="*60)

np.random.seed(99)
y_rand = pd.Series(np.random.permutation(y_loso_all), index=df_loso.index)
rand_results = []

# Pour gagner du temps sur le sanity check raw data, on ne fait qu'un fold ou on simplifie
# Ici on fait la boucle complète mais c'est lourd.
for i, (train_idx, test_idx) in enumerate(logo.split(X_loso_imp, y_rand, groups=groups_loso)):
    site_test_name = groups_loso[test_idx][0]
    X_tr, X_te = X_loso_imp.iloc[train_idx], X_loso_imp.iloc[test_idx]
    y_tr, y_te = y_rand.iloc[train_idx], y_rand.iloc[test_idx]
    c_tr, c_te = df_loso.iloc[train_idx][conf_vars], df_loso.iloc[test_idx][conf_vars]
    
    if CONF_METHOD == "regression":
        X_tr_cl, X_te_cl = residualize_confounds(X_tr, X_te, c_tr, c_te)
    else:
        X_tr_cl, X_te_cl = X_tr, X_te
        
    pipeline = get_model_raw_mlp()
    pipeline.fit(X_tr_cl, y_tr)
    y_prob = pipeline.predict_proba(X_te_cl)[:, 1]
    
    auc_r = roc_auc_score(y_te, y_prob)
    print(f"   Site {site_test_name} | AUC Random: {auc_r:.3f}")
    rand_results.append({"site": site_test_name, "auc_random": auc_r})

print("\n--- RÉSULTATS RANDOM CHECK ---")
df_rand = pd.DataFrame(rand_results)
print(f"Moyenne AUC Random : {df_rand['auc_random'].mean():.3f}")

# ============================================================
# 6) TOP CONNEXIONS
# ============================================================
print("\n" + "="*60)
print("VISUALISATION TEXTUELLE (Top Importances)")
print("="*60)

if feature_weights_list:
    df_w = pd.concat(feature_weights_list, ignore_index=True)
    df_cons = df_w.groupby("feature").agg({"weight": "mean"})
    df_cons = df_cons.sort_values(by="weight", ascending=False)
    
    print("--- TOP 10 CONNEXIONS (Importance MLP RAW) ---")
    print(df_cons.head(10))
else:
    print("Aucun poids récupéré.")

--- Chargement : ../datasets/schaefcomb_Wang2023Simple_dfschizo.tsv ---


/tmp/ipykernel_1827759/2063167766.py:24: DtypeWarning: Columns (93968) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all = pd.read_csv(DATA_PATH, sep="\t")



DÉBUT LOSO (RAW DATA - NO PCA) | MLP Neural Network
ATTENTION : Le temps de calcul sera plus long.

🔹 SITE TEST : ds000030
   Hidden: (10,), Alpha: 1 | AUC CV: 0.673 | AUC Ext: 0.632

🔹 SITE TEST : ds004302
   Hidden: (30,), Alpha: 1 | AUC CV: 0.569 | AUC Ext: 0.600

🔹 SITE TEST : ds_cobre
   Hidden: (10,), Alpha: 50 | AUC CV: 0.649 | AUC Ext: 0.774

--- RÉSULTATS RAW MLP ---
       site  auc_int  auc_ext  delta  brier
0  ds000030    0.673    0.632  0.041  0.298
1  ds004302    0.569    0.600 -0.031  0.275
2  ds_cobre    0.649    0.774 -0.125  0.288
Moyenne AUC Externe : 0.669

DÉBUT SANITY CHECK
   Site ds000030 | AUC Random: 0.470
   Site ds004302 | AUC Random: 0.542
   Site ds_cobre | AUC Random: 0.502

--- RÉSULTATS RANDOM CHECK ---
Moyenne AUC Random : 0.504

VISUALISATION TEXTUELLE (Top Importances)
--- TOP 10 CONNEXIONS (Importance MLP RAW) ---
                weight
feature               
corr_185_317  0.116164
corr_390_426  0.114082
corr_332_336  0.112814
corr_368_370  0.11227